**PALEOS H₂O Equation of State: Lookup Table Generation**

This notebook generates a pre-computed thermodynamic property table for H₂O from the
AQUA equation of state (Haldemann et al. 2020), with the corrected entropy and internal
energy from Mazevet et al. (2019), and derived heat capacities and thermal expansion.

Unlike the iron and MgSiO₃ table notebooks, there is no grid resolution optimisation
section: the source data is itself a table, so re-gridding at a fixed 150 points per
decade is sufficient. The AQUA grid (~70 ppd in P, ~100 ppd in T) is coarser than 150
ppd, so our output grid re-samples the interpolated
AQUA surface at a uniform resolution that matches the iron and MgSiO₃ tables.

**Contents:**
1. **Table Generation** — Creates the final lookup table with all seven thermodynamic
   properties plus phase information.

**Author:** Mara Attia  
**Date:** March 2026  

In [ ]:
import numpy as np
import warnings
import datetime
import os

# Import PALEOS water EoS module
from paleos import water_eos as h2o

print("PALEOS H₂O EoS Lookup Table Generator")
print("======================================")

---
# Lookup Table Generation

This section generates the final lookup table with:
- All seven thermodynamic properties (ρ, u, s, C_P, C_V, α, ∇_ad)
- Phase identification for each grid point
- Corrected entropy and internal energy (Mazevet et al. 2019 sign fix)
- Derived heat capacities and thermal expansion from AQUA table quantities
- Comprehensive header with usage information

The P–T domain matches the AQUA table range. The grid resolution is fixed
at 150 points per decade, consistent with the iron and MgSiO₃ tables.

In [ ]:
# ============================================================================
# USER INPUT CELL - Modify these parameters as needed
# ============================================================================

# Path to the AQUA P-T table
AQUA_TABLE_PATH = "../../AQUA/Tables/aqua_eos_pt_v1_0.dat"

# Pressure range [Pa]
TABLE_P_MIN = 1e5       # 1 bar
TABLE_P_MAX = 1e14      # 100 TPa

# Temperature range [K]
TABLE_T_MIN = 300.0     # 300 K
TABLE_T_MAX = 1e5       # 100,000 K

# Grid resolution: points per decade (fixed)
TABLE_PPD = 150

# Output filename
OUTPUT_FILENAME = "paleos_water_eos_table_pt.dat"

# ============================================================================

In [ ]:
def make_log_grid(x_min, x_max, ppd):
    """
    Create a log-uniform grid with specified points per decade.
    
    Parameters
    ----------
    x_min : float
        Minimum value (linear scale)
    x_max : float
        Maximum value (linear scale)
    ppd : int
        Points per decade
    
    Returns
    -------
    log_x : array
        log10(x) grid values
    n_points : int
        Total number of points
    """
    log_min = np.log10(x_min)
    log_max = np.log10(x_max)
    n_decades = log_max - log_min
    n_points = int(np.ceil(n_decades * ppd)) + 1
    log_x = np.linspace(log_min, log_max, n_points)
    return log_x, n_points

In [ ]:
# Build the output grid
log_P_table, TABLE_N_P = make_log_grid(TABLE_P_MIN, TABLE_P_MAX, TABLE_PPD)
log_T_table, TABLE_N_T = make_log_grid(TABLE_T_MIN, TABLE_T_MAX, TABLE_PPD)
P_table = 10**log_P_table
T_table = 10**log_T_table

n_decades_P = np.log10(TABLE_P_MAX) - np.log10(TABLE_P_MIN)
n_decades_T = np.log10(TABLE_T_MAX) - np.log10(TABLE_T_MIN)

print("Table generation parameters:")
print(f"  AQUA table: {AQUA_TABLE_PATH}")
print(f"  P range: [{TABLE_P_MIN:.0e}, {TABLE_P_MAX:.0e}] Pa  ({n_decades_P:.1f} decades)")
print(f"  T range: [{TABLE_T_MIN:.0f}, {TABLE_T_MAX:.0e}] K  ({n_decades_T:.2f} decades)")
print(f"  Resolution: {TABLE_PPD} points per decade")
print(f"  Grid size: {TABLE_N_P} x {TABLE_N_T} = {TABLE_N_P * TABLE_N_T:,} points")
print(f"  Output: {OUTPUT_FILENAME}")

In [ ]:
# Load the AQUA table into a Haldemann20 instance
print(f"Loading AQUA table from {AQUA_TABLE_PATH}...")
eos = h2o.Haldemann20(AQUA_TABLE_PATH)
print(f"  AQUA grid: {eos.n_P} x {eos.n_T} points")
print(f"  AQUA log₁₀(P) range: [{eos.log10_P_grid[0]:.2f}, {eos.log10_P_grid[-1]:.2f}]")
print(f"  AQUA log₁₀(T) range: [{eos.log10_T_grid[0]:.2f}, {eos.log10_T_grid[-1]:.2f}]")
print("Done.")

In [ ]:
def compute_all_properties(eos, P, T):
    """
    Compute all thermodynamic properties at a single (P, T) point.
    
    Returns
    -------
    dict with keys:
        'rho'      : density [kg/m³]
        'u'        : specific internal energy [J/kg]  (corrected)
        's'        : specific entropy [J/(kg·K)]      (corrected)
        'cp'       : isobaric heat capacity [J/(kg·K)]
        'cv'       : isochoric heat capacity [J/(kg·K)]
        'alpha'    : thermal expansion [K⁻¹]
        'nabla_ad' : adiabatic gradient [dimensionless]
        'phase'    : phase string
    """
    try:
        rho = eos.density(P, T)
    except Exception:
        rho = np.nan

    try:
        u = eos.specific_internal_energy(P, T)
    except Exception:
        u = np.nan

    try:
        s = eos.specific_entropy(P, T)
    except Exception:
        s = np.nan

    try:
        cp = eos.isobaric_heat_capacity(P, T)
    except Exception:
        cp = np.nan

    try:
        cv = eos.isochoric_heat_capacity(P, T)
    except Exception:
        cv = np.nan

    try:
        alpha = eos.thermal_expansion(P, T)
    except Exception:
        alpha = np.nan

    try:
        nabla_ad = eos.adiabatic_gradient(P, T)
    except Exception:
        nabla_ad = np.nan

    try:
        phase = eos.phase(P, T)
    except Exception:
        phase = 'unknown'

    return {
        'rho': rho,
        'u': u,
        's': s,
        'cp': cp,
        'cv': cv,
        'alpha': alpha,
        'nabla_ad': nabla_ad,
        'phase': phase
    }

In [ ]:
# Generate the table
print("Generating lookup table...")
print("=" * 60)

# Storage for all data
table_data = []

total_points = TABLE_N_P * TABLE_N_T
n_computed = 0
n_failed = 0

print(f"Computing {total_points:,} grid points...")

for i, P in enumerate(P_table):
    if (i + 1) % 100 == 0 or i == 0:
        print(f"  Progress: {i+1}/{TABLE_N_P} pressure slices "
              f"({100*(i+1)/TABLE_N_P:.0f}%)")

    for j, T in enumerate(T_table):
        props = compute_all_properties(eos, P, T)

        # Check for failures
        n_nan = sum(1 for k in ['rho', 'u', 's', 'cp', 'cv', 'alpha', 'nabla_ad']
                    if np.isnan(props[k]))
        if n_nan > 0:
            n_failed += 1

        table_data.append({
            'P': P,
            'T': T,
            **props
        })
        n_computed += 1

print(f"\nComputation complete:")
print(f"  Total points: {n_computed:,}")
print(f"  Points with NaN values: {n_failed:,} ({100*n_failed/n_computed:.2f}%)")

In [ ]:
def generate_table_header():
    """
    Generate comprehensive header for the lookup table.
    """
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S UTC")

    header = f"""\
# ==============================================================================
# PALEOS H₂O Equation of State Lookup Table
# ==============================================================================
#
# Description:
#   Pre-computed thermodynamic properties for H₂O across the complete planetary
#   phase diagram, re-gridded from the AQUA equation of state (Haldemann et al.
#   2020) with corrected entropy and internal energy (Mazevet et al. 2019 sign
#   fix) and derived heat capacities and thermal expansion.
#
# Generation Info:
#   Generated: {timestamp}
#   Resolution: {TABLE_PPD} points per decade (log-uniform in P and T)
#   Grid size: {TABLE_N_P} x {TABLE_N_T} = {TABLE_N_P * TABLE_N_T:,} points
#
# Domain:
#   Pressure:    [{TABLE_P_MIN:.8e}, {TABLE_P_MAX:.8e}] Pa
#   Temperature: [{TABLE_T_MIN:.8e}, {TABLE_T_MAX:.8e}] K
#
# Corrections Applied:
#   The Mazevet et al. (2019) Helmholtz free energy F_T contains a sign
#   error in the published Eq. (13).  The correction F_T,shift is
#   propagated analytically to entropy and internal energy.  Pressure
#   and density are unaffected (F_T,shift is volume-independent).
#
# Derived Quantities:
#   alpha    = -(1/rho)(drho/dT)_P   [numerical derivative]
#   cp       = alpha * P / (rho * nabla_ad)
#   cv       = cp^2 / (cp + T * alpha^2 * w^2)  [w = speed of sound]
#
# Interpolation Notes:
#   - Perform bilinear interpolation in (log10(P), log10(T)) space
#   - Interpolation across phase boundaries will have larger errors;
#     consider using the phase column to detect and handle these cases
#
# Phase Codes:
#   solid-ice-Ih    : Ice-Ih
#   solid-ice-II    : Ice-II
#   solid-ice-III   : Ice-III
#   solid-ice-V     : Ice-V
#   solid-ice-VI    : Ice-VI
#   solid-ice-VII   : Ice-VII
#   solid-ice-X     : Ice-X
#   vapour          : Vapor
#   liquid          : Liquid
#   supercritical   : Supercritical/superionic
#
# EoS Sources (AQUA composite, Haldemann et al. 2020):
#   Ice-Ih          : Feistel & Wagner (2006)
#   Ice-II/III/V/VI : Journaux et al. (2020)
#   Ice-VII/X       : French & Redmer (2015)
#   Liquid/gas      : Wagner & Pruß (2002), Brown (2018)
#   Ideal gas       : Gordon & McBride (1994)
#   Supercritical   : Mazevet et al. (2019)
#
# Columns:
#   1. P         - Pressure [Pa]
#   2. T         - Temperature [K]
#   3. rho       - Density [kg/m³]
#   4. u         - Specific internal energy [J/kg]  (corrected)
#   5. s         - Specific entropy [J/(kg·K)]      (corrected)
#   6. cp        - Isobaric heat capacity [J/(kg·K)]
#   7. cv        - Isochoric heat capacity [J/(kg·K)]
#   8. alpha     - Thermal expansion coefficient [K⁻¹]
#   9. nabla_ad  - Adiabatic gradient (d ln T / d ln P)_S [dimensionless]
#  10. phase     - Phase identifier (see Phase Codes above)
#
# Usage Example (Python):
#   import numpy as np
#   from scipy.interpolate import RegularGridInterpolator
#
#   cols = ['P', 'T', 'rho', 'u', 's', 'cp', 'cv', 'alpha', 'nabla_ad', 'phase']
#   data = np.genfromtxt('water_eos_table.dat', names=cols, dtype=None,
#                        encoding='utf-8')
#
#   n_P, n_T = {TABLE_N_P}, {TABLE_N_T}
#   log_P = np.log10(data['P'].reshape(n_P, n_T)[:, 0])
#   log_T = np.log10(data['T'].reshape(n_P, n_T)[0, :])
#   rho = data['rho'].reshape(n_P, n_T)
#
#   interp_rho = RegularGridInterpolator((log_P, log_T), rho)
#
#   P_query, T_query = 50e9, 2000  # 50 GPa, 2000 K
#   rho_interp = interp_rho([[np.log10(P_query), np.log10(T_query)]])[0]
#
# ==============================================================================
"""
    return header

In [ ]:
# Write the table to file
print(f"Writing table to {OUTPUT_FILENAME}...")

with open(OUTPUT_FILENAME, 'w') as f:
    # Write header
    f.write(generate_table_header())

    # Column names
    f.write("# P T rho u s cp cv alpha nabla_ad phase\n")

    # Data rows
    for row in table_data:
        def fmt(x):
            if np.isnan(x):
                return "NaN".rjust(16)
            else:
                return f"{x:.8e}"

        line = (f"{fmt(row['P'])} {fmt(row['T'])} {fmt(row['rho'])} "
                f"{fmt(row['u'])} {fmt(row['s'])} {fmt(row['cp'])} "
                f"{fmt(row['cv'])} {fmt(row['alpha'])} {fmt(row['nabla_ad'])} "
                f"{row['phase']}\n")
        f.write(line)

print(f"Done! Table written to {OUTPUT_FILENAME}")

file_size = os.path.getsize(OUTPUT_FILENAME)
print(f"File size: {file_size / 1e6:.1f} MB")

In [ ]:
# Verification: Quick sanity check of the generated table
print("\nVerification: Reading back table and checking consistency...")
print("=" * 60)

# Read the table
numeric_cols = ['P', 'T', 'rho', 'u', 's', 'cp', 'cv', 'alpha', 'nabla_ad']
data = np.genfromtxt(OUTPUT_FILENAME, names=numeric_cols + ['phase'],
                     dtype=None, encoding='utf-8')

print(f"Table shape: {len(data)} rows")
print(f"Expected:    {TABLE_N_P * TABLE_N_T} rows")
print(f"Columns: {data.dtype.names}")

# Check for NaN counts
print("\nNaN counts by column:")
for col in numeric_cols:
    vals = data[col].astype(float)
    n_nan = np.sum(np.isnan(vals))
    print(f"  {col:10s}: {n_nan:6d} ({100*n_nan/len(data):.2f}%)")

# Phase distribution
phases, counts = np.unique(data['phase'], return_counts=True)
print("\nPhase distribution:")
for phase, count in zip(phases, counts):
    print(f"  {phase:20s}: {count:6d} ({100*count/len(data):.1f}%)")

# Value ranges
print("\nValue ranges (excluding NaN):")
for col in ['rho', 'u', 's', 'cp', 'cv', 'alpha', 'nabla_ad']:
    vals = data[col].astype(float)
    valid = vals[~np.isnan(vals)]
    if len(valid) > 0:
        print(f"  {col:10s}: [{valid.min():.3e}, {valid.max():.3e}]")